# EDA Base - Transactions

This notebook loads transactions from TimescaleDB and provides a base exploratory analysis.

## Setup

In [ ]:
import os
import socket

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()

TIMESCALE_HOST = os.getenv("TIMESCALE_HOST", "localhost")
TIMESCALE_PORT = int(os.getenv("TIMESCALE_PORT", "5433"))
TIMESCALE_USER = os.getenv("TIMESCALE_USER", "postgres")
TIMESCALE_PASSWORD = os.getenv("TIMESCALE_PASSWORD", "postgres")
TIMESCALE_DB = os.getenv("TIMESCALE_DB", "timescaledb")

try:
    socket.getaddrinfo(TIMESCALE_HOST, TIMESCALE_PORT)
except socket.gaierror:
    TIMESCALE_HOST = "localhost"
    if TIMESCALE_PORT == 5432:
        TIMESCALE_PORT = 5433
    print("TimescaleDB host not resolvable. Using localhost.")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


def create_connection():
    return psycopg2.connect(
        host=TIMESCALE_HOST,
        port=TIMESCALE_PORT,
        user=TIMESCALE_USER,
        password=TIMESCALE_PASSWORD,
        dbname=TIMESCALE_DB,
    )


def load_transactions():
    query = """
        SELECT
            transaction_id,
            user_id,
            merchant_id,
            merchant_category,
            amount::double precision AS amount,
            country,
            device_type,
            ip_hash,
            timestamp,
            is_fraud,
            model_score,
            latency_ms
        FROM public.transactions
    """
    with create_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            rows = cursor.fetchall()
            columns = [column[0] for column in cursor.description]
    return pd.DataFrame(rows, columns=columns)


transactions = load_transactions()
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], utc=True)
transactions["is_fraud"] = transactions["is_fraud"].astype(bool)

transactions.head()

## Class distribution

In [ ]:
class_counts = transactions["is_fraud"].value_counts().reindex([False, True], fill_value=0)
class_summary = pd.DataFrame(
    {
        "count": class_counts,
        "percent": (class_counts / class_counts.sum() * 100).round(2),
    }
)
class_summary.index = class_summary.index.map({False: "legit", True: "fraud"})

print(class_summary.to_string())

ax = sns.barplot(x=class_summary.index, y=class_summary["count"], palette="deep")
ax.set_title("Transaction class distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Transaction count")
plt.show()

## Amount distribution

In [ ]:
amount_stats = transactions.groupby("is_fraud").agg(
    mean=("amount", "mean"),
    median=("amount", "median"),
    std=("amount", "std"),
    min=("amount", "min"),
    max=("amount", "max"),
    p95=("amount", lambda values: values.quantile(0.95)),
    p99=("amount", lambda values: values.quantile(0.99)),
)
amount_stats.index = amount_stats.index.map({False: "legit", True: "fraud"})

print(amount_stats.round(2).to_string())

positive_amounts = transactions[transactions["amount"] > 0]
if len(positive_amounts) < len(transactions):
    print("Note: non-positive amounts excluded from log-scale histogram.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, label in zip(axes, [False, True], strict=False):
    subset = positive_amounts[positive_amounts["is_fraud"] == label]
    sns.histplot(subset["amount"], bins=50, ax=ax, color="#4c72b0")
    ax.set_xscale("log")
    ax.set_title(f"Amount distribution - {'fraud' if label else 'legit'}")
    ax.set_xlabel("Amount (log scale)")
    ax.set_ylabel("Transaction count")

plt.tight_layout()
plt.show()

## Transactions by hour

In [ ]:
transactions["transaction_hour"] = transactions["timestamp"].dt.hour

hourly_counts = (
    transactions.groupby("transaction_hour")["transaction_id"]
    .count()
    .reindex(range(24), fill_value=0)
)
hourly_fraud_rate = (
    transactions.groupby("transaction_hour")["is_fraud"].mean().reindex(range(24), fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=hourly_counts.index, y=hourly_counts.values, ax=axes[0], color="#4c72b0")
axes[0].set_title("Transaction volume by hour (UTC)")
axes[0].set_xlabel("Hour of day")
axes[0].set_ylabel("Transaction count")

sns.lineplot(x=hourly_fraud_rate.index, y=hourly_fraud_rate.values, ax=axes[1], marker="o")
axes[1].set_title("Fraud rate by hour (UTC)")
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Fraud rate")

plt.tight_layout()
plt.show()

## Transactions by country

In [ ]:
country_summary = (
    transactions.groupby("country")
    .agg(count=("transaction_id", "count"), fraud_rate=("is_fraud", "mean"))
    .sort_values("count", ascending=False)
)
top_countries = country_summary.head(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=top_countries, x="country", y="count", ax=axes[0], color="#4c72b0")
axes[0].set_title("Top countries by transaction volume")
axes[0].set_xlabel("Country")
axes[0].set_ylabel("Transaction count")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=top_countries, x="country", y="fraud_rate", ax=axes[1], color="#dd8452")
axes[1].set_title("Fraud rate by country (top volume)")
axes[1].set_xlabel("Country")
axes[1].set_ylabel("Fraud rate")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Transactions by merchant category

In [ ]:
category_summary = (
    transactions.groupby("merchant_category")
    .agg(count=("transaction_id", "count"), fraud_rate=("is_fraud", "mean"))
    .sort_values("count", ascending=False)
)
top_categories = category_summary.head(10).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=top_categories, x="merchant_category", y="count", ax=axes[0], color="#4c72b0")
axes[0].set_title("Top merchant categories by volume")
axes[0].set_xlabel("Merchant category")
axes[0].set_ylabel("Transaction count")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=top_categories, x="merchant_category", y="fraud_rate", ax=axes[1], color="#dd8452")
axes[1].set_title("Fraud rate by merchant category (top volume)")
axes[1].set_xlabel("Merchant category")
axes[1].set_ylabel("Fraud rate")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Transactions by device type

In [ ]:
device_summary = (
    transactions.groupby("device_type")
    .agg(count=("transaction_id", "count"), fraud_rate=("is_fraud", "mean"))
    .sort_values("count", ascending=False)
)
device_summary = device_summary.reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=device_summary, x="device_type", y="count", ax=axes[0], color="#4c72b0")
axes[0].set_title("Transaction volume by device type")
axes[0].set_xlabel("Device type")
axes[0].set_ylabel("Transaction count")

sns.barplot(data=device_summary, x="device_type", y="fraud_rate", ax=axes[1], color="#dd8452")
axes[1].set_title("Fraud rate by device type")
axes[1].set_xlabel("Device type")
axes[1].set_ylabel("Fraud rate")

plt.tight_layout()
plt.show()